<a href="https://colab.research.google.com/github/Doson-062/vietnamese-contextual-intent/blob/main/notebooks/1.0-dts-phobert-b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cấu hình B — PhoBERT fine-tune, chỉ câu hiện tại

Đồ án: *Phân loại ý định tiếng Việt trong môi trường có ngữ cảnh*. Notebook dùng lại `src/dataset.py` và `src/modeling/evaluate.py` của repo, nên **cùng mẫu, cùng cách chấm** với cấu hình A.

**Cách chạy:**
1. `Runtime → Change runtime type → T4 GPU`.
2. `Runtime → Run all`. Mất khoảng 15–30 phút cho 3 seed.
3. Ô cuối tải file kết quả (`.zip`) về máy. Giải nén vào thư mục `reports/` của dự án.

**Đổi cấu hình** ở ô 2:
- `K = 0` → cấu hình **B** (chỉ câu hiện tại).
- `K = 1, 2, 3` hoặc `None` (toàn bộ lịch sử) → cấu hình **C**.
- `BACKBONE = "uitnlp/visobert"` → so sánh backbone (ViSoBERT không cần tách từ).

In [ ]:
# 1. Lấy code + dữ liệu từ GitHub và cài thư viện (mỗi phiên Colab chạy lại ô này)
import os, sys
REPO = "/content/vietnamese-contextual-intent"
if not os.path.exists(REPO):
    !git clone -q https://github.com/Doson-062/vietnamese-contextual-intent.git {REPO}
else:
    !git -C {REPO} pull -q
os.chdir(REPO)
sys.path.insert(0, REPO)
!pip install -q py_vncorenlp
!java -version
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Cấu hình thí nghiệm
BACKBONE = "vinai/phobert-base-v2"   # hoặc "uitnlp/visobert" để so sánh backbone
SEGMENT = BACKBONE.startswith("vinai/phobert")  # PhoBERT bắt buộc tách từ (RDRSegmenter); ViSoBERT thì không
K = 0                                # 0 = cấu hình B; 1, 2, 3 hoặc None (toàn bộ lịch sử) = cấu hình C
SEEDS = [42, 43, 44]                 # chạy nhiều seed, báo cáo trung bình ± độ lệch chuẩn
EPOCHS, LR, BATCH = 5, 2e-5, 32
MAX_LEN = 128 if K == 0 else 256     # PhoBERT tối đa 256 token
RUN_NAME = ("baseline_b" if K == 0 else f"config_c_k{K}") + "_" + BACKBONE.split("/")[-1]
print(RUN_NAME)

In [ ]:
# 3. Đọc mẫu bằng đúng code của repo (bỏ LOAI và đa nhãn, vẫn giữ chúng làm ngữ cảnh)
from src.config import LABELS, REPORTS_DIR
from src.dataset import load_samples
from src.modeling.evaluate import evaluate, format_report, accuracy_by_context
data = {s: load_samples(s, k=K) for s in ("train", "dev", "test")}
for s, v in data.items():
    print(s, len(v), "mẫu")

In [ ]:
# 4. Tách từ bằng RDRSegmenter (VnCoreNLP) — đúng công cụ VinAI dùng khi tiền huấn luyện PhoBERT
texts = {x["text"] for v in data.values() for x in v} | {u for v in data.values() for x in v for _, u in x["context"]}
if SEGMENT:
    import py_vncorenlp
    SEG_DIR = "/content/vncorenlp"
    os.makedirs(SEG_DIR, exist_ok=True)
    if not os.path.exists(f"{SEG_DIR}/models"):
        py_vncorenlp.download_model(save_dir=SEG_DIR)
    seg = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=SEG_DIR)
    os.chdir(REPO)  # py_vncorenlp tự đổi thư mục làm việc, đổi lại
    cache = {t: " ".join(seg.word_segment(t)) for t in texts}
else:
    cache = {t: t for t in texts}
print(len(cache), "câu khác nhau. Ví dụ:")
for t in list(cache)[:3]:
    print(" ", t, "->", cache[t])

In [ ]:
# 5. Mã hóa đầu vào. B: chỉ câu hiện tại. C: cặp (lịch sử, câu hiện tại); lịch sử quá dài thì cắt phần CŨ nhất
import torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding,
                          Trainer, TrainingArguments, set_seed)
tok = AutoTokenizer.from_pretrained(BACKBONE)
tok.truncation_side = "left"
lab2id = {l: i for i, l in enumerate(LABELS)}
SPK = {"USER": "Khách", "SYSTEM": "NV"}

def encode(split):
    v = data[split]
    cur = [cache[x["text"]] for x in v]
    if K == 0:
        enc = tok(cur, truncation=True, max_length=MAX_LEN)
    else:
        hist = [" ".join(f"{SPK[s]}: {cache[u]}" for s, u in x["context"]) for x in v]
        enc = tok(hist, cur, truncation="only_first", max_length=MAX_LEN)
    enc["labels"] = [lab2id[x["label"]] for x in v]
    return Dataset.from_dict(dict(enc))

ds = {s: encode(s) for s in data}
print(tok.decode(ds["train"][0]["input_ids"]))

In [ ]:
# 6. Huấn luyện: học trên train, chọn epoch tốt nhất theo macro-F1 trên dev, chấm test một lần — lặp cho từng seed
def dev_metrics(p):
    m = evaluate([LABELS[i] for i in p.label_ids], [LABELS[i] for i in p.predictions.argmax(-1)])
    return {"macro_f1": m["macro_f1"], "accuracy": m["accuracy"]}

y_true = [x["label"] for x in data["test"]]
runs = []
# 10% số bước đầu tăng dần learning rate (warmup_steps có ở mọi phiên bản transformers; warmup_ratio đã bị bỏ ở bản mới)
WARMUP = int(0.1 * EPOCHS * ((len(ds["train"]) + BATCH - 1) // BATCH))
for seed in SEEDS:
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        BACKBONE, num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id=lab2id)
    args = TrainingArguments(
        output_dir=f"/content/ckpt/{RUN_NAME}_{seed}", num_train_epochs=EPOCHS, learning_rate=LR,
        per_device_train_batch_size=BATCH, per_device_eval_batch_size=64, warmup_steps=WARMUP, weight_decay=0.01,
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1, load_best_model_at_end=True,
        metric_for_best_model="macro_f1", fp16=torch.cuda.is_available(), report_to="none", seed=seed, logging_steps=50)
    trainer = Trainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["dev"],
                      data_collator=DataCollatorWithPadding(tok), compute_metrics=dev_metrics)
    trainer.train()
    y_pred = [LABELS[i] for i in trainer.predict(ds["test"]).predictions.argmax(-1)]
    m = evaluate(y_true, y_pred)
    m["by_context"] = accuracy_by_context([x["text"] for x in data["test"]], y_true, y_pred)
    m["best_dev_macro_f1"] = trainer.state.best_metric
    runs.append({"seed": seed, "test": m})
    print(f"seed {seed}: test acc {m['accuracy']:.4f} | macro-F1 {m['macro_f1']:.4f} | "
          f"câu cần ngữ cảnh {m['by_context']['can_ngu_canh']['accuracy']:.1%}")
    del model, trainer
    torch.cuda.empty_cache()

In [ ]:
# 7. Tổng hợp nhiều seed và lưu kết quả vào reports/<RUN_NAME>/
import json, statistics as st
PICK = {"accuracy": lambda m: m["accuracy"], "macro_f1": lambda m: m["macro_f1"],
        "co_nhac_domain": lambda m: m["by_context"]["co_nhac_domain"]["accuracy"],
        "can_ngu_canh": lambda m: m["by_context"]["can_ngu_canh"]["accuracy"]}
summary = {}
for key, f in PICK.items():
    vals = [f(r["test"]) for r in runs]
    summary[key] = {"mean": st.mean(vals), "std": st.stdev(vals) if len(vals) > 1 else 0.0}

out = REPORTS_DIR / RUN_NAME
out.mkdir(parents=True, exist_ok=True)
config = {"backbone": BACKBONE, "k": K, "segment": SEGMENT, "seeds": SEEDS, "epochs": EPOCHS,
          "lr": LR, "batch": BATCH, "max_len": MAX_LEN}
json.dump({"config": config, "summary": summary, "runs": runs},
          open(out / "metrics.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
lines = [f"{RUN_NAME}: {config}", "",
         *[f"{k:<16}{v['mean']:.4f} ± {v['std']:.4f}" for k, v in summary.items()], "",
         f"=== Chi tiết seed {runs[0]['seed']} ===", format_report(runs[0]["test"])]
(out / "report.txt").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines))

In [ ]:
# 8. Tải kết quả về máy (giải nén vào thư mục reports/ của dự án)
!cd /content/vietnamese-contextual-intent/reports && zip -qr /content/{RUN_NAME}.zip {RUN_NAME}
from google.colab import files
files.download(f"/content/{RUN_NAME}.zip")